# Laboratório: Swarm

**Dupla:**

- Nome 1:
- Nome 2:

Neste laboratório deverá ser construído, com LangChain e o `langgraph-swarm`, um swarm em que os agentes transferem a conversa entre si por handoffs, sem supervisor. O tema é escolhido pela dupla, e o sistema deverá ser testado com conversas de vários turnos, em que o assunto muda de área no meio do caminho. Cada seção descreve o que se espera dela e traz um espaço para o código, e podem ser criadas quantas células forem precisas.

In [ ]:
# No Google Colab, descomente e rode uma vez.

# !pip install -q langchain langchain-openai langgraph langgraph-swarm

# import os
# from google.colab import userdata

# os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

In [ ]:
import pandas as pd
from IPython.display import Image
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain.messages import ToolMessage
from langchain.tools import ToolRuntime, tool
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command
from langgraph_swarm import create_handoff_tool, create_swarm

## Tema

Descrição do serviço, de quem conversaria com ele e de como se sabe que um pedido foi bem atendido. O swarm serve a temas em que o usuário conversa direto com especialistas por vários turnos, como um atendimento com áreas diferentes, uma agência de viagens ou um assistente de estudos. O tema deve ter pelo menos três áreas, pedidos que passam por mais de uma delas e pelo menos um caso em que uma área precisa do que a outra descobriu. As ferramentas de cada área são definidas aqui, e podem devolver dados fixos ou consultar uma API pública.

## Agentes e handoffs

Pelo menos três agentes criados com `create_agent`, cada um com `name`, a própria mensagem de sistema, as próprias ferramentas e um papel que os outros não cumprem. Os handoffs são criados com `create_handoff_tool`, com uma `description` que diga quando transferir, e a mensagem de sistema de cada agente diz o que ele resolve e para quem vai o que está fora da sua área. O modelo deve ser criado com `model_kwargs={"parallel_tool_calls": False}`, porque uma chamada feita junto com um handoff ficaria sem resposta.

## Swarm

Swarm montado com `create_swarm`, com o `default_active_agent` e um checkpointer, seguido do desenho do grafo. Deve ser explicado por que esse agente recebe o primeiro turno e, se algum agente ficou sem handoff para outro, por que essa transferência não deve acontecer. Uma conversa de pelo menos três turnos na mesma thread deve ser mostrada, com uma linha por mensagem e o `active_agent` ao fim de cada turno, incluindo um turno que começa em um agente e termina em outro.

## Propagação de contexto

O `create_handoff_tool` propaga o histórico inteiro de quem transfere. Um dos handoffs deve ser reescrito à mão, como uma ferramenta que devolve um `Command` e leva só o par da chamada, com um resumo escrito por quem transfere no conteúdo da `ToolMessage`. O `update` também deve gravar o destino em `active_agent`, para o turno seguinte começar por ele. A mesma conversa deve ser rodada com as duas versões, comparando o que o agente de destino recebeu, se ele resolveu o pedido e os tokens de entrada.

## Avaliação

Conjunto de pelo menos oito conversas, como uma lista de dicionários, cada uma com os turnos, o agente que deveria terminar ativo e o resultado esperado. O conjunto deve incluir conversas de uma área só, conversas que passam por duas áreas no mesmo turno, conversas em que o assunto muda de um turno para o outro e pelo menos um pedido que nenhum agente deveria atender. Cada conversa roda em uma thread própria, com um `recursion_limit`, registrando:

- se o agente ativo ao fim da conversa é o esperado;
- se o resultado está certo, por uma verificação em Python sobre as chamadas de ferramenta ou por um LLM como juiz com saída estruturada, conferido à mão em uma amostra;
- o número de handoffs e os tokens de entrada.

## Análise

Discussão sobre os resultados:

- Em quais conversas o pedido chegou ao agente errado ou ficou preso em um agente? O que faltava, um handoff ou uma regra na mensagem de sistema?
- Houve transferências desnecessárias ou vai e volta entre dois agentes? Em qual conversa, e o que causou?
- O que o handoff com resumo economizou em tokens, e alguma informação se perdeu com ele?
- Um supervisor atenderia melhor esse tema? O que mudaria na resposta ao usuário, no turno seguinte e no custo?